# Hand Calculation Formulas for Parameters, Activations, FLOPs, and Memory

> In previous sections we built MiniGPT, read SmolLM2's config.json, and estimated the ZeRO memory of a 7B model on 8 GPUs. All these numbers share an underlying set of formulas: how to count parameters, how to count activations, how to count FLOPs, how to count memory. Once these formulas are internalized, you can instantly compute the training/inference cost of any model given its config.
>
> This section works through all of it systematically. We first derive closed-form formulas for parameters, activations, and FLOPs term by term, then validate them with PyTorch. Next we lay out two ledgers — inference memory and training memory — covering KV cache, AdamW states, and gradient checkpointing. Finally, we discuss which formulas need to be rewritten for MoE.


Nearly every interview question and every capacity-planning exercise in LLM engineering eventually reduces to the same question: given hidden size $D$, number of layers $L$, vocabulary size $V$, batch size $B$, and sequence length $S$, how many FLOPs does one forward pass of this model cost? How much memory does training take? How large is the KV cache at inference?

These questions share a common property: they can all be computed with paper and pencil. The formulas are not complicated; the key is to clearly map each matrix's shape to the number of multiply-add operations it entails. This notebook uses a unified set of symbols:

- $D$ = hidden size, the uniform width inside a Block
- $L$ = number of layers
- $V$ = vocabulary size
- $H$ = number of attention heads, $K$ = number of KV heads (under GQA, $K < H$), $d_h = D / H$ is the dimension per head
- $F$ = FFN intermediate width (under SwiGLU configuration, typically $F \approx 8D/3$)
- $B$ = batch size, $S$ = sequence length
- $P$ = total number of parameters in the model

Each section below follows the same routine: first derive a closed-form formula with these symbols, then measure it with `sum(p.numel())` or `torch.cuda.memory_allocated()`, and finally compare predicted versus measured.


## 1. Parameters: From a Single Layer to the Whole Model

Let us first look at the trainable weights inside a Transformer Block. A Llama-style Block has three parts: Attention (four projections Q/K/V/O), FFN (three projections gate/up/down, corresponding to SwiGLU), and two RMSNorms. Writing out the shape of each weight makes the parameter count clear.


In [ ]:
import torch
import torch.nn as nn

# Use SmolLM2-135M's real config for demonstration
D = 576      # hidden size
H = 9        # num_attention_heads
K = 3        # num_key_value_heads (GQA)
L = 30       # num_hidden_layers
V = 49152    # vocab_size
FF = 1536    # intermediate_size (FFN)
d_h = D // H  # head_dim = 64

# --- Attention four projections ---
# Standard MHA: Q,K,V are all [D, H*d_h]
# GQA:          K,V are [D, K*d_h]; only Q is [D, H*d_h]
q_params = D * (H * d_h)           # = D * D
k_params = D * (K * d_h)           # smaller under GQA
v_params = D * (K * d_h)
o_params = (H * d_h) * D           # = D * D

attn_params = q_params + k_params + v_params + o_params
print("=== Single-layer Attention parameters (GQA) ===")
print(f"Q: [{D}, {H*d_h}]  -> {q_params:,}")
print(f"K: [{D}, {K*d_h}]  -> {k_params:,}  (GQA shrunk to {K/H:.2f} of Q)")
print(f"V: [{D}, {K*d_h}]  -> {v_params:,}")
print(f"O: [{H*d_h}, {D}]  -> {o_params:,}")
print(f"Total: {attn_params:,}")

# Under standard MHA (K=H), the four projections sum to approximately 4D^2
mha_attn = 4 * D * D
print(f"\nIf reverted to MHA (K=H={H}), Attention parameters ~= 4D^2 = {mha_attn:,}")
print(f"  Formula 4D^2 gives {4*D*D:,} at D={D}, identical to the exact MHA value.")


The simplified form of Attention under MHA is $4D^2$. Next, the FFN. A SwiGLU FFN has three matrices: gate, up, and down, each involving one multiplication between $D$ and $F$.

| Projection | Shape | Parameters |
|:---|:---|:---|
| gate | $[D, F]$ | $DF$ |
| up   | $[D, F]$ | $DF$ |
| down | $[F, D]$ | $DF$ |

The three terms sum to $3DF$. There is an engineering rationale here: modern models commonly choose $F = 8D/3$ because it keeps the total FFN parameter count near "twice the Attention's $4D^2$" ($3D \cdot \tfrac{8D}{3} = 8D^2 \approx 2 \times 4D^2$); $F$ is then often rounded up so that it is divisible by 64.


In [ ]:
# --- FFN three projections (SwiGLU) ---
gate_p = D * FF
up_p   = D * FF
down_p = FF * D
ffn_params = gate_p + up_p + down_p

print("=== Single-layer FFN parameters (SwiGLU) ===")
print(f"gate: [{D}, {FF}]  -> {gate_p:,}")
print(f"up:   [{D}, {FF}]  -> {up_p:,}")
print(f"down: [{FF}, {D}]  -> {down_p:,}")
print(f"Total: {ffn_params:,}  (formula 3DF = 3*{D}*{FF} = {3*D*FF:,})")

# Origin of 8D/3: make FFN params ~= 2x Attention params
F_ideal = 8 * D / 3
print(f"\n'Ideal' F = 8D/3 = {F_ideal:.1f}")
print(f"Actual config F = {FF} (rounded up to a multiple of 64: {((int(F_ideal) + 63) // 64) * 64})")
print(f"Substituted back: 3D*(8D/3) = 8D^2 = {8*D*D:,}  ~= twice the Attention's {4*D*D:,}")


**Per-layer subtotal and the global parameter count**

Each layer's RMSNorm has only $D$ trainable scale parameters (no bias); two RMSNorms add up to $2D$. So a single Block's parameter count is:

$$P_{\text{layer}} = \underbrace{4D^2}_{\text{Attention}} + \underbrace{3DF}_{\text{FFN}} + \underbrace{2D}_{\text{Norm}} \approx 12D^2 \;\;\text{(substituting } F=8D/3 \text{)}$$

Adding the head and tail embeddings ($V \cdot D$) and unembedding ($V \cdot D$) globally gives $2VD$ if not tied, or just $VD$ if tied. A commonly used estimate:

$$P \approx 2VD + 12LD^2$$


In [ ]:
# --- Per-layer and global formulas ---
norm_params = 2 * D
per_layer = attn_params + ffn_params + norm_params
print(f"=== Per-layer subtotal ===")
print(f"Attention: {attn_params:,}  (formula 4D^2 ~= {4*D*D:,})")
print(f"FFN:       {ffn_params:,}  (formula 3DF = {3*D*FF:,})")
print(f"Norm:      {norm_params:,}")
print(f"Per layer: {per_layer:,}  (formula 12D^2 ~= {12*D*D:,}, error {abs(per_layer-12*D*D)/per_layer*100:.2f}%)")

# --- Global total parameter count formula ---
total_formula = 2 * V * D + L * 12 * D * D   # estimate
total_exact = V * D + L * per_layer + D       # exact value with tied word embeddings

print(f"\n=== Global total parameters ===")
print(f"Embedding (V*D):        {V*D:>12,}")
print(f"{L} layers Block (L*per-layer): {L*per_layer:>12,}")
print(f"final Norm:             {D:>12,}")
print(f"Exact total (tied):     {total_exact:>12,}  ~= {total_exact/1e6:.1f}M")
print(f"Estimate 2VD+12LD^2:    {total_formula:>12,}  (overestimate, because it ignores tying and GQA)")


In [ ]:
# --- Build the full model with PyTorch and validate against sum(p.numel()) ---
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    """Llama-style Block: RMSNorm -> GQA Attention -> RMSNorm -> SwiGLU FFN"""
    def __init__(self, d, h, kv, d_head, ff_dim):
        super().__init__()
        self.attn_norm = nn.RMSNorm(d)
        self.ffn_norm  = nn.RMSNorm(d)
        self.q_proj = nn.Linear(d, h * d_head, bias=False)
        self.k_proj = nn.Linear(d, kv * d_head, bias=False)
        self.v_proj = nn.Linear(d, kv * d_head, bias=False)
        self.o_proj = nn.Linear(h * d_head, d, bias=False)
        self.gate = nn.Linear(d, ff_dim, bias=False)
        self.up   = nn.Linear(d, ff_dim, bias=False)
        self.down = nn.Linear(ff_dim, d, bias=False)
        self.h, self.kv, self.d_head = h, kv, d_head

    def forward(self, x):
        # Simplified implementation: no RoPE, no causal mask
        residual = x
        x = self.attn_norm(x)
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.d_head).transpose(1, 2)
        groups = self.h // self.kv
        k = k.repeat_interleave(groups, dim=1)
        v = v.repeat_interleave(groups, dim=1)
        scale = self.d_head ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        x = residual + self.o_proj(out)
        residual = x
        x = self.ffn_norm(x)
        x = residual + self.down(F.silu(self.gate(x)) * self.up(x))
        return x

class MiniLM(nn.Module):
    def __init__(self, V, D, L, H, K, FF):
        super().__init__()
        d_h = D // H
        self.embed = nn.Embedding(V, D)
        self.blocks = nn.ModuleList(
            [TransformerBlock(D, H, K, d_h, FF) for _ in range(L)])
        self.final_norm = nn.RMSNorm(D)
    def forward(self, idx):
        x = self.embed(idx)
        for blk in self.blocks: x = blk(x)
        x = self.final_norm(x)
        return F.linear(x, self.embed.weight)   # tied

torch.manual_seed(42)
model = MiniLM(V, D, L, H, K, FF)
total_real = sum(p.numel() for p in model.parameters())

print("=== Formula vs PyTorch measurement ===")
print(f"Formula prediction: {total_exact:,}")
print(f"PyTorch:            {total_real:,}")
print(f"Error:              {abs(total_exact - total_real):,}  ({'exact match' if total_exact==total_real else 'needs checking'})")


## 2. Activation Memory: What Each Forward Layer Stores Temporarily

Parameter count concerns "how many weights in total"; activation concerns "how much memory the intermediate tensors occupy during forward". Activations correlate strongly with batch size and sequence length, whereas parameter count does not — this is the biggest difference between the two.

Listing all tensors produced during a single Block's forward and summing them up term by term:

**Attention activations** (MHA, without FlashAttention)

| Tensor | Shape | Size |
|:---|:---|:---|
| RMSNorm input | $[B, S, D]$ | $BSD$ |
| RMSNorm output | $[B, S, D]$ | $BSD$ |
| Q, K, V | $[B, S, D]$ x 3 | $3BSD$ |
| Attention scores $QK^\top$ | $[B, H, S, S]$ | $BHS^2 = BS^2 \cdot (H)$ |
| Attention output | $[B, S, D]$ | $BSD$ |

Summing the above gives roughly $6BSD + BHS^2$ (where $BHS^2 = BS^2 \cdot H = BNS^2$ and $N=H$ is the number of heads). The second term $BNS^2$ is the total cost of the $S \times S$ attention matrix.


In [ ]:
# --- Demonstrate the actual shapes of each Attention tensor ---
B, S = 2, 16
x = torch.randn(B, S, D)

# Simulate the Attention forward and record each intermediate tensor's bytes (FP32 = 4 bytes)
def show(name, t):
    print(f"  {name:<22} shape={list(t.shape)}  bytes={t.nelement()*4:,}")

print(f"=== Attention activation list (B={B}, S={S}, D={D}, H={H}) ===")
norm_in = x                                         # RMSNorm input
show("norm_in", norm_in)
norm_out = norm_in                                  # RMSNorm output
show("norm_out", norm_out)
q = torch.randn(B, S, H * d_h)
k = torch.randn(B, S, K * d_h)
v = torch.randn(B, S, K * d_h)
show("Q/K/V total", torch.cat([q, k, v], dim=-1))
scores = torch.randn(B, H, S, S)
show("attn_scores QK^T", scores)
attn_out = torch.randn(B, S, D)
show("attn_out", attn_out)

# Formula check
attn_act_formula = 6 * B * S * D + B * H * S * S
attn_act_bytes = (B*S*D + B*S*D + B*S*(H+2*K)*d_h + B*H*S*S + B*S*D) * 4
print(f"\nFormula 6BSD + BHS^2 = {attn_act_formula:,} elements")
print(f"  = {attn_act_formula*4/1024:.1f} KB (FP32)")


**FFN activations**

| Tensor | Shape | Size |
|:---|:---|:---|
| RMSNorm input | $[B, S, D]$ | $BSD$ |
| gate output | $[B, S, F]$ | $BSF$ |
| up output | $[B, S, F]$ | $BSF$ |
| down output | $[B, S, D]$ | $BSD$ |

These add up to roughly $2BSD + 2BSF$; substituting $F = 8D/3$ gives $2BSD + 2BS \cdot \tfrac{8D}{3} = 2BSD + \tfrac{16}{3}BSD \approx 6.33BSD$, often simplified to $8BSD$.

**Per-layer activation total**: Attention's $6BSD + BNS^2$ plus FFN's $8BSD$ yields the classic $14BSD + BNS^2$.


In [ ]:
# --- Per-layer activation formula 14BSD + BNS^2 ---
B, S = 8, 4096
per_layer_act = 14 * B * S * D + B * H * S * S

print(f"=== Per-layer activation formula (B={B}, S={S}, D={D}, H={H}) ===")
print(f"Attention: 6BSD + BHS^2 = {6*B*S*D + B*H*S*S:,} elements")
print(f"FFN:       8BSD         = {8*B*S*D:,} elements")
print(f"Per layer: 14BSD+BHS^2  = {per_layer_act:,} elements")
print(f"  = {per_layer_act * 4 / 1e9:.2f} GB (FP32)")
print(f"  = {per_layer_act * 2 / 1e9:.2f} GB (BF16)")

# All L layers
total_act = L * per_layer_act
print(f"\n{L} layers total: {total_act:,} elements")
print(f"  = {total_act * 4 / 1e9:.2f} GB (FP32)")
print(f"  = {total_act * 2 / 1e9:.2f} GB (BF16)")

# See which term dominates
linear_part = L * 14 * B * S * D
quad_part = L * B * H * S * S
print(f"\n=== Dominant term analysis ===")
print(f"Linear term 14*BSD*L = {linear_part*4/1e9:.2f} GB ({linear_part/per_layer_act/L*100:.1f}% of per-layer)")
print(f"Quadratic term BHS^2*L = {quad_part*4/1e9:.2f} GB ({quad_part/per_layer_act/L*100:.1f}% of per-layer)")
print(f"  At S={S}, the S^2 term is already close to the linear term.")


## 3. FLOPs: How Many Multiply-Adds a Forward Pass Performs

FLOPs (floating point operations) is the basic unit for measuring compute. One multiplication and one addition each count as 1 op, so the FLOPs of a matrix multiplication $[m, k] \times [k, n]$ is $2mkn$.

**Matrix multiplication FLOPs formula**: for $Y = XW$, where $X \in \mathbb{R}^{m \times k}$, $W \in \mathbb{R}^{k \times n}$, and output $Y \in \mathbb{R}^{m \times n}$, each output element requires $k$ multiplications + $k$ additions = $2k$ ops, giving $2mkn$ ops in total.

Applying this formula to each matmul in Attention and FFN, we sum the FLOPs term by term.


In [ ]:
# === Attention forward FLOPs ===
# Each matmul: [m, k] x [k, n] -> 2*m*k*n
# Here m = B*S; k and n depend on the specific projection

m = B * S   # total number of tokens

# Q projection: [B*S, D] x [D, D] -> 2*B*S*D*D
flops_q = 2 * m * D * (H * d_h)
# K projection: [B*S, D] x [D, K*d_h] -> 2*B*S*D*K*d_h
flops_k = 2 * m * D * (K * d_h)
# V projection: same as K
flops_v = 2 * m * D * (K * d_h)
# QK^T: [B, H, S, d_h] x [B, H, d_h, S] -> 2*B*H*S*S*d_h = 2*B*S^2*D
flops_qkt = 2 * B * H * S * S * d_h
# AV:   [B, H, S, S] x [B, H, S, d_h] -> 2*B*H*S*S*d_h = 2*B*S^2*D
flops_av = 2 * B * H * S * S * d_h
# O projection: [B*S, D] x [D, D] -> 2*B*S*D*D
flops_o = 2 * m * D * (H * d_h)

attn_flops = flops_q + flops_k + flops_v + flops_qkt + flops_av + flops_o
print(f"=== Attention forward FLOPs (MHA, B={B}, S={S}, D={D}) ===")
print(f"Q projection: {flops_q:,}")
print(f"K projection: {flops_k:,}  (GQA, K-heads={K})")
print(f"V projection: {flops_v:,}")
print(f"QK^T:         {flops_qkt:,}  <- S^2 term")
print(f"AV:           {flops_av:,}  <- S^2 term")
print(f"O projection: {flops_o:,}")
print(f"Total:        {attn_flops:,}  ~= {attn_flops/1e9:.2f} GFLOPs")
print(f"\nFormula 8BSD^2 + 4BS^2D (MHA) = {8*B*S*D*D + 4*B*S*S*D:,}, error is small")


In [ ]:
# === FFN forward FLOPs (SwiGLU: gate, up, down, three matmuls) ===
flops_gate = 2 * m * D * FF
flops_up   = 2 * m * D * FF
flops_down = 2 * m * FF * D
ffn_flops = flops_gate + flops_up + flops_down

print(f"=== FFN forward FLOPs ===")
print(f"gate: {flops_gate:,}")
print(f"up:   {flops_up:,}")
print(f"down: {flops_down:,}")
print(f"Total: {ffn_flops:,}  (formula 6BSDF = {6*B*S*D*FF:,})")
print(f"Substituting F=8D/3: 16BSD^2 = {16*B*S*D*D:,}  ~= {ffn_flops/1e9:.2f} GFLOPs")

# === Per layer + global ===
per_layer_flops = attn_flops + ffn_flops
total_flops = L * per_layer_flops + 2 * B * S * D * V  # add unembedding

# Closed form: 2BSD(12LD + 2LS + V)
closed_form = 2 * B * S * D * (12 * L * D + 2 * L * S + V)

print(f"\n=== Global forward FLOPs ===")
print(f"Per layer (Attention+FFN): {per_layer_flops:,}  ~= {per_layer_flops/1e9:.2f} GFLOPs")
print(f"unembedding [B*S,D]x[D,V]: {2*B*S*D*V:,}")
print(f"{L} layers + unembedding: {total_flops:,}  ~= {total_flops/1e12:.2f} TFLOPs")
print(f"\nClosed-form 2BSD(12LD + 2LS + V) = {closed_form:,}")
print(f"  Error: {abs(total_flops-closed_form)/total_flops*100:.2f}%")


## 4. Backpropagation: FLOPs Are About Twice the Forward

Backpropagation computes two kinds of gradients for every op: the gradient with respect to the weights $\partial L / \partial W$ and the gradient with respect to the input $\partial L / \partial X$. Each gradient is a matmul, so the total FLOPs of the backward pass are about twice those of the forward pass.

The commonly cited "6x rule of thumb" comes from here: forward 1 part + backward 2 parts = 3 parts of FLOPs; factoring in optimizer updates and other per-step overheads, we multiply by another 2 as a safety margin.

**Estimating single-step training FLOPs**: $\text{FLOPs}_{\text{step}} \approx 6 \cdot 2BSD \cdot (12LD + 2LS + V) \approx 6PD$ (when $S$ is not large, simplified to 6 times the product of parameter count and token count).


In [ ]:
# === Training FLOPs rule of thumb: 6 * P * tokens ===
# Total FLOPs to train SmolLM2-135M on 2T tokens
P_smolLM = 135e6
tokens_smolLM = 2e12
train_flops_smolLM = 6 * P_smolLM * tokens_smolLM

print(f"=== SmolLM2-135M training FLOPs ===")
print(f"Parameters P = {P_smolLM/1e6:.0f}M")
print(f"Training tokens = {tokens_smolLM/1e12:.0f}T")
print(f"Total training FLOPs = 6PD = {train_flops_smolLM:.2e}")
print(f"  = {train_flops_smolLM/1e21:.2f} ZFLOPs")

# 7B model trained on 1.5T tokens (LLaMA-1 configuration)
P_7b = 7e9
tokens_7b = 1.5e12
train_flops_7b = 6 * P_7b * tokens_7b
print(f"\n=== 7B model training FLOPs ===")
print(f"Parameters P = {P_7b/1e9:.0f}B, tokens = {tokens_7b/1e12:.1f}T")
print(f"Total training FLOPs = 6PD = {train_flops_7b:.2e} = {train_flops_7b/1e21:.2f} ZFLOPs")
print(f"Theoretical wall-clock on 2048 A100s (312 TFLOPs/s): {train_flops_7b/(2048*312e12)/86400:.1f} days")


## 5. Inference Memory: Weights + KV Cache + Peak Activations

Inference memory has three components:

1. **Weights**: $P$ parameters, $2P$ bytes under FP16, $P$ bytes under INT8
2. **KV cache**: each token stores two tensors, K and V, in every layer
3. **Peak activations**: temporary tensors during a single layer's forward (released immediately after forward completes)

KV cache memory depends on batch size, sequence length, number of layers, and the number of KV heads. The KV size per token per layer is $2 \cdot K \cdot d_h$ (one copy each for K and V); multiplying by batch, sequence length, number of layers, and bytes/element gives the total KV cache memory.

$$\text{KV cache} = B \cdot S \cdot K \cdot d_h \cdot L \cdot 2 \cdot \text{bytes/element}$$


In [ ]:
# === KV cache memory by hand (7B model) ===
# Typical 7B model configuration
D_7b = 4096
L_7b = 32
H_7b = 32
K_7b = 32        # LLaMA-1 uses MHA, K=H
d_h_7b = D_7b // H_7b   # 128
V_7b = 32000

# Estimate parameter count
P_7b_formula = 2 * V_7b * D_7b + 12 * L_7b * D_7b * D_7b
print(f"=== 7B model ===")
print(f"Formula P ~= 2VD + 12LD^2 = {P_7b_formula/1e9:.2f}B")

# FP16 weight memory
weights_fp16 = 2 * P_7b_formula
print(f"FP16 weights: {weights_fp16/1e9:.2f} GB")

# KV cache: various batch and seq combinations
scenarios = [
    ("single user S=2048", 1,    2048),
    ("single user S=8192", 1,    8192),
    ("B=8, S=2048",        8,    2048),
    ("B=32, S=2048",       32,   2048),
    ("B=32, S=8192",       32,   8192),
]

print(f"\n=== KV cache memory (FP16, KV-heads={K_7b}, head_dim={d_h_7b}) ===")
print(f"{'Scenario':<22} {'KV cache (GB)':>15}")
for name, b, s in scenarios:
    kv_elems = b * s * K_7b * d_h_7b * L_7b * 2  # x2 for K and V
    kv_bytes = kv_elems * 2   # FP16
    print(f"{name:<22} {kv_bytes/1e9:>15.2f}")


In [ ]:
# === Three dominant regimes of inference memory ===
# X-axis: batch size; compare weights, KV cache, activations

import numpy as np

S_fixed = 2048
B_range = np.array([1, 2, 4, 8, 16, 32, 64, 128])

weights_gb = np.full_like(B_range, 2 * P_7b_formula / 1e9, dtype=float)
kv_gb = B_range * S_fixed * K_7b * d_h_7b * L_7b * 2 * 2 / 1e9
# Peak activations (rough: per-layer BSD x ~10 + BNS^2)
act_gb = (10 * B_range * S_fixed * D_7b + B_range * H_7b * S_fixed**2) * 2 / 1e9

print(f"{'B':>5} {'weights':>10} {'KV cache':>10} {'activations':>12} {'dominant':>12}")
print("-" * 55)
for i, b in enumerate(B_range):
    parts = {'weights': weights_gb[i], 'KV': kv_gb[i], 'act': act_gb[i]}
    dom = max(parts, key=parts.get)
    print(f"{b:>5} {weights_gb[i]:>9.2f}G {kv_gb[i]:>9.2f}G {act_gb[i]:>11.2f}G {dom:>12}")

print("\n=== Three dominant regimes ===")
print("(1) Small batch + short sequence: weights dominate; KV/activations are tiny")
print("(2) Long S: the S^2 term dominates activations, and KV cache grows linearly")
print("(3) Large batch: KV cache grows linearly; both BSD and BHS^2 in activations grow")


## 6. Training Memory: The Origin of 16P + Activations

Training memory is far more complex than inference. Under AdamW + mixed precision, each parameter is accompanied by the following items:

| Item | bytes/param | Note |
|:---|:---|:---|
| FP32 master weights | 4P | master weights under mixed precision |
| BF16 transient weights | 2P | low-precision copy used during forward |
| AdamW first moment $m$ | 4P | FP32 accumulation |
| AdamW second moment $v$ | 4P | FP32 accumulation |
| FP32 gradients | 4P | BF16 compute during backward, FP32 accumulation |
| Subtotal | **16P** | this is the "fixed overhead" |

Adding activations gives the full training memory. The activation component depends on $B, S, D, L$ via the formula $L \cdot (14BSD + BNS^2)$ and can be substantially reduced by gradient checkpointing.


In [ ]:
# === 7B model training memory by hand ===
P = P_7b_formula

# Fixed overhead 16P
fp32_master  = 4 * P
bf16_weights = 2 * P
adam_m       = 4 * P
adam_v       = 4 * P
fp32_grad    = 4 * P    # BF16 compute during backward, FP32 accumulation
fixed = fp32_master + bf16_weights + adam_m + adam_v + fp32_grad

print(f"=== 7B training fixed overhead ===")
print(f"FP32 master weights: {fp32_master/1e9:>6.2f} GB")
print(f"BF16 weight copy:    {bf16_weights/1e9:>6.2f} GB")
print(f"AdamW m:             {adam_m/1e9:>6.2f} GB")
print(f"AdamW v:             {adam_v/1e9:>6.2f} GB")
print(f"FP32 gradients:      {fp32_grad/1e9:>6.2f} GB")
print(f"Fixed subtotal:      {fixed/1e9:>6.2f} GB  (= 16P)")

# Activations: L*(14BSD + BHS^2) (BF16 = 2 bytes)
B_train, S_train = 8, 4096
D, H, L = D_7b, H_7b, L_7b
per_layer_act = 14 * B_train * S_train * D + B_train * H * S_train * S_train
act_total = L * per_layer_act * 2  # BF16

print(f"\n=== Activation memory (B={B_train}, S={S_train}, BF16) ===")
print(f"Per layer 14BSD+BHS^2: {per_layer_act:,} elements")
print(f"L={L} layers total:    {L*per_layer_act:,} elements = {act_total/1e9:.2f} GB")

print(f"\n=== Total training memory ===")
total_train = fixed + act_total
print(f"Fixed 16P:           {fixed/1e9:>6.2f} GB ({fixed/total_train*100:.0f}%)")
print(f"Activations:         {act_total/1e9:>6.2f} GB ({act_total/total_train*100:.0f}%)")
print(f"Total:               {total_train/1e9:>6.2f} GB")
print(f"Single A100 80GB:    {'does not fit' if total_train/1e9 > 80 else 'fits'}")


## 7. Gradient Checkpointing: Trading Compute for Memory

Activations explode at long sequence lengths and can no longer fit on a single GPU. The idea behind gradient checkpointing: during forward, do not keep every layer's intermediate activations; only keep the inputs of a few "checkpoint" layers. During backward, recompute the forward from the checkpoint to obtain the needed activations on the fly.

The cost is one extra forward, and memory drops from $O(L)$ to $O(\sqrt{L})$. Specifically, divide the $L$ layers into $\sqrt{L}$ segments, each with $\sqrt{L}$ layers, and store only the boundary input of each segment — store $\sqrt{L}$ boundaries, recompute $\sqrt{L}$ layers per segment during backward, for a total of $O(\sqrt{L})$ storage + $O(L)$ extra compute.

PyTorch provides the `torch.utils.checkpoint.checkpoint` API.


In [ ]:
# === Gradient checkpointing empirical comparison (simulated on CPU) ===
import torch.utils.checkpoint as cp

class CheckpointedBlock(nn.Module):
    """Wrap a TransformerBlock; use gradient checkpointing during forward"""
    def __init__(self, block):
        super().__init__()
        self.block = block
    def forward(self, x):
        # Input must have requires_grad=True to allow recompute
        return cp.checkpoint(self.block, x, use_reentrant=False)

# Build two equivalent MiniLMs: one without checkpointing, one with checkpointing per layer
torch.manual_seed(0)
model_plain = MiniLM(V, D_7b // 4, 8, 8, 8, D_7b // 4 * 8 // 3)
# Shrink the size for CPU testing: D=1024, L=8
model_plain = MiniLM(V=1000, D=256, L=8, H=4, K=4, FF=512)

torch.manual_seed(0)
model_ckpt = MiniLM(V=1000, D=256, L=8, H=4, K=4, FF=512)
for i, blk in enumerate(model_ckpt.blocks):
    model_ckpt.blocks[i] = CheckpointedBlock(blk)

# Measure memory (allocated is unavailable on CPU; using torch.cuda requires a GPU; here we only verify it runs)
idx = torch.randint(0, 1000, (2, 64))
logits_plain = model_plain(idx)
logits_ckpt = model_ckpt(idx)
print(f"Model A (plain) output shape:       {list(logits_plain.shape)}")
print(f"Model B (checkpoint) output shape:  {list(logits_ckpt.shape)}")
print(f"Same weights:                       {torch.allclose(model_plain.embed.weight, model_ckpt.embed.weight)}")

# Use formulas to estimate activation memory under the two schemes
L_test = 8
B_test, S_test, D_test = 2, 64, 256
H_test = 4
per_layer_act_test = 14 * B_test * S_test * D_test + B_test * H_test * S_test * S_test

# Plain: store all L layers
plain_act = L_test * per_layer_act_test
# Checkpoint (sqrt strategy): store only sqrt(L) boundaries, recompute during backward
sqrt_L = int(L_test ** 0.5)
ckpt_act = sqrt_L * per_layer_act_test  # rough: storage drops to sqrt(L)/L

print(f"\n=== Activation memory comparison (L={L_test}, B={B_test}, S={S_test}) ===")
print(f"Plain:             L * per-layer   = {plain_act:,} elements")
print(f"Checkpoint (sqrt(L)): sqrt(L) * per-layer = {ckpt_act:,} elements")
print(f"Savings:           {1 - ckpt_act/plain_act:.0%}")
print(f"Cost: forward computed once more (~+33% FLOPs)")


## 8. MoE: Total Parameters vs Active Parameters

In a MoE (Mixture of Experts) model, the FFN is replaced by N parallel expert FFNs, and each token picks its top-k experts through a router. This changes how parameters and FLOPs are counted.

**Total parameters** vs **active parameters**:

- **Total parameters**: all experts' FFN parameters are counted. Mixtral 8x7B has 8 experts, each the size of a dense 7B's FFN; together with attention/embedding, total parameters are about 46.7B.
- **Active parameters**: each token activates only top-2 experts, so the parameters that are actually "live" during forward are just attention + 2 expert FFNs, about 13B.

**FLOPs** should be computed using active parameters, because only the top-k experts' FFNs actually perform matmuls.

**KV cache is unchanged**: MoE only swaps the FFN for experts; attention is still a dense shared structure, so the KV cache formula stays the same.


In [ ]:
# === Mixtral 8x7B parameter count by hand ===
# Assume each expert ~= the FFN configuration of LLaMA-7B
D_mix = 4096
L_mix = 32
H_mix = 32
K_mix = 8        # GQA in Mixtral
d_h_mix = D_mix // H_mix
FF_mix = 14336   # Mixtral FFN dim
V_mix = 32000
E = 8            # 8 experts
top_k = 2

# Per-layer attention (dense portion)
attn_per_layer = 4 * D_mix * D_mix  # MHA approximation

# Per-layer FFN: E experts, each 3*D*FF
ffn_all_experts = E * 3 * D_mix * FF_mix
ffn_active = top_k * 3 * D_mix * FF_mix   # top-k activated

per_layer_total = attn_per_layer + ffn_all_experts
per_layer_active = attn_per_layer + ffn_active

P_total = V_mix * D_mix + L_mix * per_layer_total + D_mix   # unembedding counted separately when tie=False
P_active = V_mix * D_mix + L_mix * per_layer_active + D_mix

print(f"=== Mixtral 8x7B ===")
print(f"Number of experts E = {E}, top-k = {top_k}")
print(f"Per-layer Attention:       {attn_per_layer/1e6:.0f}M")
print(f"Per-layer FFN (all {E} experts):    {ffn_all_experts/1e6:.0f}M")
print(f"Per-layer FFN ({top_k} activated):  {ffn_active/1e6:.0f}M")
print(f"\nTotal params P_total:   {P_total/1e9:.1f}B  (paper reports 46.7B)")
print(f"Active params P_active: {P_active/1e9:.1f}B  (paper reports ~13B)")
print(f"Activation ratio = {P_active/P_total:.1%}")

# FLOPs use P_active
tokens = 1e12
flops_moe = 6 * P_active * tokens
flops_dense_equiv = 6 * P_total * tokens
print(f"\nTraining on {tokens/1e12:.0f}T tokens:")
print(f"  MoE actual FLOPs (6*P_active*D):     {flops_moe/1e21:.2f} ZFLOPs")
print(f"  Equivalent dense FLOPs (6*P_total*D): {flops_dense_equiv/1e21:.2f} ZFLOPs")
print(f"  MoE trains a larger model with 1/{flops_dense_equiv/flops_moe:.1f} of the compute")


In [ ]:
# === KV cache is unchanged under MoE ===
# Because attention remains dense and shared; experts only replace the FFN
B_infer, S_infer = 1, 8192

# Mixtral's KV cache (same formula as dense 7B)
kv_mixtral = B_infer * S_infer * K_mix * d_h_mix * L_mix * 2 * 2  # x2 K/V, x2 FP16
print(f"=== KV cache comparison (B={B_infer}, S={S_infer}, FP16) ===")
print(f"Mixtral 8x7B: {kv_mixtral/1e9:.2f} GB")
print(f"dense 7B:     {B_infer * S_infer * K_7b * d_h_7b * L_7b * 2 * 2 / 1e9:.2f} GB")
print(f"Both share the identical formula: B*S*K*d_h*L*2*2")
print(f"\nKey observation: MoE increases total parameters, but inference FLOPs and KV cache are computed by active params")
print(f"  This is the core advantage of MoE over large dense models: higher quality per unit of compute")


## Summary

Confirm you understand these formulas:

- [ ] Single-layer Attention params ~= $4D^2$ (MHA); FFN (SwiGLU) ~= $3DF$; substituting $F=8D/3$ gives FFN ~= $8D^2$
- [ ] Global parameter formula: $P \approx 2VD + 12LD^2$
- [ ] Single-layer activations ~= $14BSD + BNS^2$; the quadratic term comes from the $S \times S$ attention matrix
- [ ] Single-layer forward FLOPs ~= $8BSD^2 + 4BS^2D + 16BSD^2$; global ~= $2BSD(12LD + 2LS + V)$
- [ ] Backward FLOPs ~= 2 x forward; training rule of thumb 6PD
- [ ] KV cache size: $B \cdot S \cdot K \cdot d_h \cdot L \cdot 2 \cdot \text{bytes/element}$
- [ ] Training memory fixed overhead = 16P (FP32 master + BF16 transient + AdamW m/v + FP32 grad)
- [ ] Gradient checkpointing reduces activations from $O(L)$ to $O(\sqrt{L})$, at the cost of recomputing the forward
- [ ] MoE total parameters count all experts; active parameters / FLOPs / KV cache use the top-k activated experts


## Exercises

> You may use AI to help explain ideas, but it is not recommended to have AI "complete the exercise for you".

**Exercise 1: Estimate LLaMA-2 70B's parameter count and FP16 weight memory**

LLaMA-2 70B configuration: $D=8192$, $L=80$, $H=64$, $K=8$ (GQA), $V=32000$. Use the formula $P \approx 2VD + 12LD^2$ to compute the parameter count, then compute the FP16 weight memory.

Hint: parameter count ~= $2VD + 12LD^2$; under FP16 each parameter takes 2 bytes.


In [ ]:
# Exercise 1: LLaMA-2 70B parameter count and weight memory
D_l2 = 8192
L_l2 = 80
V_l2 = 32000

# TODO: estimate parameter count using the formula
P_l2 = None  # 2*V*D + 12*L*D^2

# TODO: compute FP16 weight memory (bytes)
weights_bytes_l2 = None

# Verify
assert P_l2 is not None, "Please compute P_l2 with the formula first"
assert weights_bytes_l2 is not None, "Please compute weights_bytes_l2 first"
expected_P = 2 * V_l2 * D_l2 + 12 * L_l2 * D_l2 ** 2
assert abs(P_l2 - expected_P) / expected_P < 0.01, f"Parameter count should be about {expected_P/1e9:.1f}B"
expected_bytes = 2 * expected_P
assert weights_bytes_l2 == expected_bytes, f"Memory should be {expected_bytes/1e9:.1f} GB"
print(f"Exercise 1 passed:")
print(f"  LLaMA-2 70B parameter count ~= {P_l2/1e9:.1f}B")
print(f"  FP16 weight memory ~= {weights_bytes_l2/1e9:.1f} GB")


**Exercise 2: Compute training memory for a 13B model at batch=8, seq=4096**

13B model ($D=5120$, $L=40$, $H=40$, MHA), trained with AdamW + BF16 mixed precision. batch=8, seq=4096. Compute: fixed overhead (16P), activations (BF16), and total memory.

Hint: fixed = 16P bytes; activations = $L \cdot (14BSD + BHS^2)$ elements x 2 bytes.


In [ ]:
# Exercise 2: 13B model training memory
P_13b = 13e9     # Simplified: use the number 13B
B_ex, S_ex = 8, 4096
D_ex, H_ex, L_ex = 5120, 40, 40

# TODO: fixed overhead (bytes)
fixed_bytes = None  # 16 * P

# TODO: activations (bytes, BF16 = 2 bytes/element)
act_elements = None  # L * (14*B*S*D + B*H*S^2)
act_bytes = None     # act_elements * 2

# TODO: total memory (GB)
total_gb = None

assert fixed_bytes is not None and act_elements is not None and total_gb is not None
assert fixed_bytes == 16 * P_13b, "Fixed overhead should be 16P"
assert act_elements == L_ex * (14 * B_ex * S_ex * D_ex + B_ex * H_ex * S_ex * S_ex)
print(f"Exercise 2 passed:")
print(f"  Fixed overhead: {fixed_bytes/1e9:.1f} GB")
print(f"  Activations:    {act_bytes/1e9:.1f} GB")
print(f"  Total memory:   {total_gb:.1f} GB")
print(f"  Needs {int((total_gb+79)//80)} A100 80GB GPUs")


**Exercise 3: Estimate the FLOPs of a single Mixtral 8x7B prefill**

Mixtral 8x7B, top-2 routing, input $B=1$, $S=4096$. Use the active parameter count $P_{\text{active}} \approx 13B$ to estimate the forward FLOPs of a single prefill.

Hint: forward FLOPs ~= $2 \cdot P_{\text{active}} \cdot B \cdot S$ (when unembedding dominates or as a simplified estimate, this is roughly 1/3 of $6PD \cdot \text{tokens}$, i.e., forward ~= $2P_{\text{active}} \cdot$ tokens).


In [ ]:
# Exercise 3: Mixtral 8x7B prefill FLOPs
P_active_mix = 13e9
B_ex3, S_ex3 = 1, 4096
tokens_ex3 = B_ex3 * S_ex3

# TODO: forward FLOPs (simplified estimate: 2 * P_active * tokens)
forward_flops = None  # 2 * P_active_mix * tokens_ex3

assert forward_flops is not None
expected = 2 * P_active_mix * tokens_ex3
assert forward_flops == expected, f"Should equal {expected}"
print(f"Exercise 3 passed:")
print(f"  tokens = {tokens_ex3}")
print(f"  forward FLOPs ~= {forward_flops:.2e} = {forward_flops/1e12:.2f} TFLOPs")
print(f"  Note: this is the active-parameter estimate, much smaller than the dense 46.7B version")


## References

- Pope et al., [Efficient Memory Management for Large Language Model Serving with PagedAttention](https://arxiv.org/abs/2309.06180), 2023 — the classic source for KV cache memory analysis
- Korthikanti et al., [Reducing Activation Recomputation in Large Transformer Models](https://arxiv.org/abs/2205.05198), 2022 — source of Megatron-LM's activation formula $14BSD + BNS^2$
- Chen et al., [Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174), 2016 — the original gradient checkpointing paper
- Hoffmann et al., [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556), 2022 — Chinchilla scaling law, origin of 6PD
- Fedus et al., [Switch Transformers: Scaling to Trillion Parameter Models](https://arxiv.org/abs/2101.03961), 2021 — analysis of MoE parameters vs active parameters
